In [3]:
import os
import re
import string

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

from collections import Counter


In [4]:
path = r'Datasets'
csv_path = os.path.join(path,"IMDB Dataset.csv")

df = pd.read_csv(csv_path)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
print(df.shape)

(50000, 2)


In [6]:
import re

email_pattern = r"[\w.-]+@[\w.-]+\.\w+"
url_pattern = r"https?://\S+\.\S"

URL = re.compile(url_pattern)
EMAIL = re.compile(email_pattern)

In [7]:
# Removal of Stop words

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download("stopwords")
nltk.download("punkt")

stop_words = stopwords.words("english")


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [9]:
def text_clean(cell):
  if not isinstance(cell,str):
    return " "

  cell = EMAIL.sub(" ",cell)
  cell = URL.sub(" ",cell)
  cell = cell.lower()
  cell = re.sub(r"\d+"," ",cell)
  cell = re.sub(r"[^\w\s]","",cell)

  cell = [word for word in cell.split() if word not in stop_words]
  cell = " ".join(cell)

  return cell

df["clean_review"] = df["review"].apply(text_clean)

df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [10]:
df["label"] = df["sentiment"].map(
    {
        "positive":1,
        "negative":0
    }
)

df.head()

,review,sentiment,clean_review,label
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...,1
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...,1
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...,1
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...,1


In [11]:
train_texts, test_texts, train_labels,test_labels = train_test_split(
    df["clean_review"].values,
    df["label"].values,
    test_size = 0.2,
    random_state = 42
)

print("Train texts shape:",train_texts.shape)
print("Test texts shape:",test_texts.shape)

Train texts shape: (40000,)
Test texts shape: (10000,)


In [12]:
# Build Vocab

MAX_VOCAB_SIZE = 20000

word_counter = Counter()

for text in train_texts:
  words = text.split()
  word_counter.update(words)

most_common_words = word_counter.most_common(
    MAX_VOCAB_SIZE - 2
)

word_to_index = {
    "<PAD>":0,
    "<UNK>":1
}

for word, _ in most_common_words:
  word_to_index[word] = len(word_to_index)


print("Vocabulary size:",len(word_to_index))

Vocabulary size: 20000


In [13]:
print(word_to_index)

{'<PAD>': 0, '<UNK>': 1, 'br': 2, 'movie': 3, 'film': 4, 'one': 5, 'like': 6, 'good': 7, 'even': 8, 'would': 9, 'time': 10, 'really': 11, 'see': 12, 'story': 13, 'much': 14, 'well': 15, 'get': 16, 'also': 17, 'great': 18, 'bad': 19, 'people': 20, 'first': 21, 'dont': 22, 'movies': 23, 'made': 24, 'make': 25, 'could': 26, 'films': 27, 'way': 28, 'characters': 29, 'think': 30, 'watch': 31, 'many': 32, 'seen': 33, 'two': 34, 'character': 35, 'never': 36, 'love': 37, 'acting': 38, 'plot': 39, 'best': 40, 'little': 41, 'know': 42, 'show': 43, 'ever': 44, 'life': 45, 'better': 46, 'still': 47, 'end': 48, 'scene': 49, 'say': 50, 'man': 51, 'scenes': 52, 'something': 53, 'go': 54, 'im': 55, 'back': 56, 'real': 57, 'watching': 58, 'thing': 59, 'doesnt': 60, 'didnt': 61, 'actors': 62, 'years': 63, 'though': 64, 'actually': 65, 'another': 66, 'makes': 67, 'funny': 68, 'find': 69, 'nothing': 70, 'look': 71, 'going': 72, 'lot': 73, 'new': 74, 'every': 75, 'work': 76, 'old': 77, 'us': 78, 'part': 79

In [14]:
# Tokenize and pad

MAX_LENGTH = 250

def encode_text(text):
  words = text.split()

  encode = [
      word_to_index.get(word,1) for word in words
  ]

  encode = encode[:MAX_LENGTH]

  if len(encode) < MAX_LENGTH:
    encode = encode + [0] * (MAX_LENGTH - len(encode))

  return encode

example = "this movie good"

print(encode_text(example)[:10])

[1, 3, 7, 0, 0, 0, 0, 0, 0, 0]


In [15]:
class IMDBDataset(Dataset):
  def __init__(self,texts,labels):
    self.texts = texts
    self.labels = labels

  def __len__(self):
    return len(self.texts)

  def __getitem__(self,idx):
    text = self.texts[idx]
    label = self.labels[idx]

    encoded_text = encode_text(text)
    return torch.tensor(encoded_text),torch.tensor(label)

train_dataset = IMDBDataset(train_texts,train_labels)
test_dataset = IMDBDataset(test_texts,test_labels)

In [16]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)